In [1]:
import pyspark
import os
from dotenv import load_dotenv
from pathlib import Path

from pyspark.sql.functions import from_json, col, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [2]:
dotenv_path = Path("/resources/.env")
load_dotenv(dotenv_path=dotenv_path)

True

In [3]:
spark_hostname = os.getenv("SPARK_MASTER_HOST_NAME")
spark_port = os.getenv("SPARK_MASTER_PORT")
kafka_host = os.getenv("KAFKA_HOST")
kafka_topic = os.getenv("KAFKA_TOPIC_NAME")

# Load PostgreSQL configuration from environment
pg_host = os.getenv("POSTGRES_CONTAINER_NAME")
pg_dw_db = os.getenv("POSTGRES_DW_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

In [4]:
# Configure PostgreSQL JDBC URL
pg_jdbc_url = f"jdbc:postgresql://{pg_host}/{pg_dw_db}"

In [ ]:
print(spark_hostname)
#!ls /opt

In [ ]:
spark_host = f"spark://{spark_hostname}:{spark_port}"

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2 org.postgresql:postgresql:42.2.18"
)
print(spark_host)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Ruang Data Project Spark-Kafka") 
    .config("spark.streaming.stopGracefullyOnShutdown", True) 
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]") 
    .getOrCreate()
)

spark

In [5]:
spark = (
    pyspark.sql.SparkSession.builder.appName("RuangDataProjectStreaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,org.postgresql:postgresql:42.2.18")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", True)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [6]:
stream_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", f"{kafka_host}:9092")
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest")
    .load()
)

In [7]:
# Define the schema using StructType for clarity
schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("furniture", StringType()),
    StructField("color", StringType()),
    StructField("price", IntegerType()),
    StructField("ts", LongType())
])

In [12]:
# Parse the JSON data using DataFrame API
parsed_df = stream_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# Verify parsed schema
print("Parsed DataFrame Schema:")
parsed_df.printSchema()

Parsed DataFrame Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- furniture: string (nullable = true)
 |-- color: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- ts: long (nullable = true)



In [11]:
# Calculate average price
avg_price_df = parsed_df.groupBy("furniture").agg(avg("price").alias("avg_price"))

In [13]:
# Verify aggregation schema
print("Aggregated DataFrame Schema:")
avg_price_df.printSchema()

Aggregated DataFrame Schema:
root
 |-- furniture: string (nullable = true)
 |-- avg_price: double (nullable = true)



In [14]:
# Define PostgreSQL writer function
def write_to_postgresql(batch_df, batch_id):
    (batch_df.write
        .format("jdbc")
        .option("url", pg_jdbc_url)
        .option("dbtable", "furniture")  # Replace with your table name
        .option("user", pg_user)
        .option("password", pg_password)
        .option("driver", "org.postgresql.Driver")
        .mode("append")
        .save())

In [16]:
# Write the result to the console
query = parsed_df.writeStream \
    .foreachBatch(write_to_postgresql) \
    .outputMode("update") \
    .trigger(processingTime="1 second") \
    .start()
    
query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 